# AERIS — Random Forest Training & Evaluation
NASA FD001 | Current baseline (work in progress)

Run **EDA & Feature Engineering** first to export CSVs to `/content/data/processed`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
import joblib

### Step 1 — Load prepared data

What: Read the team's cleaned FD001 data. Why: Reuse preprocessing without repeating EDA.

In [ ]:
DATA_DIR = Path('/content/data/processed')  # Change this if CSV files are elsewhere.
OUTPUT_DIR = Path('/content/data/model_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
required = ['train_original.csv', 'test_original.csv', 'test_rul.csv']
missing = [f for f in required if not (DATA_DIR / f).is_file()]
if missing:
    raise FileNotFoundError(f'Missing CSV files in {DATA_DIR}: {missing}')
clean_train = pd.read_csv(DATA_DIR / 'train_original.csv')
clean_test = pd.read_csv(DATA_DIR / 'test_original.csv')
test_rul_df = pd.read_csv(DATA_DIR / 'test_rul.csv')
if 'true_rul' not in test_rul_df.columns and test_rul_df.shape[1] == 1:
    test_rul_df.columns = ['true_rul']
original_features = [c for c in clean_train.columns if c not in ['engine_id', 'RUL']]
assert len(original_features) == 18, f'Expected 18 original features, got {len(original_features)}'
assert {'engine_id', 'RUL'}.issubset(clean_train.columns)
assert {'engine_id', 'cycle'}.issubset(clean_test.columns)
assert all(c in clean_test.columns for c in original_features)
assert clean_train[original_features + ['RUL']].notna().all().all()
assert clean_test[original_features].notna().all().all()
print('Prepared data loaded:', clean_train.shape, clean_test.shape)

Prepared data loaded: (20631, 20) (13096, 19)


### Step 2 — Split by engine for tuning

What: Hold out entire engines. Why: Cycles of one engine must not leak across sets.

In [ ]:
X = clean_train[original_features]
y = clean_train['RUL']
groups = clean_train['engine_id']
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(X, y, groups=groups))
X_cv, y_cv = X.iloc[train_idx], y.iloc[train_idx]
groups_cv = groups.iloc[train_idx]
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[val_idx]))
print('Tuning rows:', len(X_cv), '| Held-out validation rows:', len(val_idx))

Tuning rows: 16561 | Held-out validation rows: 4070


### Step 3 — Tune tree depth and leaf size (original experiment 9)

What: Compare four configurations with GroupKFold.

Why: Select settings that generalize to unseen engines.

In [ ]:

# Experiment 9 — Random Forest Hyperparameter Tuning
# Four configurations to compare
configs = [
    {"max_depth": 10, "min_samples_leaf": 3},
    {"max_depth": 15, "min_samples_leaf": 3},
    {"max_depth": 20, "min_samples_leaf": 3},
    {"max_depth": 15, "min_samples_leaf": 5}
]

cv = GroupKFold(n_splits=3)
tuning_results = []

for config in configs:
    fold_scores = []

    for train_idx, val_idx in cv.split(
        X_cv, y_cv, groups=groups_cv
    ):
        model = RandomForestRegressor(
            n_estimators=50,
            max_depth=config["max_depth"],
            min_samples_leaf=config["min_samples_leaf"],
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_cv.iloc[train_idx],
            y_cv.iloc[train_idx]
        )

        predictions = model.predict(
            X_cv.iloc[val_idx]
        )

        fold_scores.append(
            mean_absolute_error(
                y_cv.iloc[val_idx],
                predictions
            )
        )

    tuning_results.append({
        "Max Depth": config["max_depth"],
        "Min Samples Leaf": config["min_samples_leaf"],
        "Mean MAE": np.mean(fold_scores),
        "MAE Std": np.std(fold_scores)
    })

tuning_df = pd.DataFrame(tuning_results)
tuning_df = tuning_df.sort_values("Mean MAE")

print(tuning_df.round(4).to_string(index=False))

print("Best configuration:")
print(tuning_df.iloc[0])

 Max Depth  Min Samples Leaf  Mean MAE  MAE Std
        10                 3   27.1785   1.2276
        15                 5   27.2783   1.2218
        15                 3   27.3289   1.2138
        20                 3   27.3674   1.2331
Best configuration:
Max Depth           10.000000
Min Samples Leaf     3.000000
Mean MAE            27.178485
MAE Std              1.227571
Name: 0, dtype: float64


### Step 4 — Compare 50 vs 100 trees (original experiment 11)

What: Check mean CV MAE for both forest sizes.

Why: Justify the selected 100 trees with experimental evidence.

In [ ]:

# Experiment 11 — Number of Trees Tuning
cv = GroupKFold(n_splits=3)
tree_results = []

for n_trees in [50, 100]:

    fold_mae = []

    for train_idx, val_idx in cv.split(
        X_cv, y_cv, groups=groups_cv
    ):

        model = RandomForestRegressor(
            n_estimators=n_trees,
            max_depth=10,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_cv.iloc[train_idx],
            y_cv.iloc[train_idx]
        )

        predictions = model.predict(
            X_cv.iloc[val_idx]
        )

        fold_mae.append(
            mean_absolute_error(
                y_cv.iloc[val_idx],
                predictions
            )
        )

    tree_results.append({
        "Trees": n_trees,
        "Mean MAE": np.mean(fold_mae),
        "MAE Std": np.std(fold_mae)
    })

tree_results_df = pd.DataFrame(tree_results)

print(tree_results_df.round(4).to_string(index=False))

best_trees = int(
    tree_results_df.loc[
        tree_results_df["Mean MAE"].idxmin(), "Trees"
    ]
)

print("Best number of trees:", best_trees)

 Trees  Mean MAE  MAE Std
    50   27.1785   1.2276
   100   27.0922   1.2571
Best number of trees: 100


### Step 5 — Train the selected Random Forest on all training engines

What: Refit using the chosen settings (100 trees, depth 10, leaf 3).

Why: Use all available training examples before NASA testing.

In [ ]:

# Final Random Forest Training — AERIS
# Prepare full training data
X_final = clean_train[original_features]
y_final = clean_train["RUL"]

# Create final model
final_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# Train on all training engines
final_rf.fit(X_final, y_final)

print("Final Random Forest trained successfully!")
print("Training samples:", X_final.shape[0])
print("Number of features:", X_final.shape[1])

Final Random Forest trained successfully!
Training samples: 20631
Number of features: 18


### Step 6 — Evaluate on NASA's separate 100 test engines

What: Predict RUL at the last observed cycle of each test engine.

Why: Measure performance on engines not used in model fitting.

In [ ]:

# Final Evaluation — NASA FD001 Test
# 1. Select the last available cycle for each test engine
last_test = (
    clean_test
    .sort_values(["engine_id", "cycle"])
    .groupby("engine_id")
    .tail(1)
    .sort_values("engine_id")
    .reset_index(drop=True)
)

# 2. Prepare the same 18 features used during training
X_test_final = last_test[original_features]

# 3. True RUL values from NASA
y_test_final = test_rul_df["true_rul"].to_numpy()

# 4. Check that every engine has one true RUL value
assert len(last_test) == len(y_test_final) == 100
assert last_test["engine_id"].tolist() == list(range(1, 101))

# 5. Predict RUL
y_pred_final = final_rf.predict(X_test_final)

# 6. Evaluate predictions
test_mae = mean_absolute_error(y_test_final, y_pred_final)
test_rmse = np.sqrt(mean_squared_error(y_test_final, y_pred_final))
test_r2 = r2_score(y_test_final, y_pred_final)

print("Final NASA FD001 Test Results")
print(f"MAE:  {test_mae:.2f} cycles")
print(f"RMSE: {test_rmse:.2f} cycles")
print(f"R2:   {test_r2:.4f}")

# 7. Show predictions for the first 10 engines
test_predictions = pd.DataFrame({
    "Engine ID": last_test["engine_id"],
    "Actual RUL": y_test_final,
    "Predicted RUL": np.round(y_pred_final, 2),
    "Absolute Error": np.round(
        np.abs(y_test_final - y_pred_final), 2
    )
})

print(test_predictions.head(10).to_string(index=False))

Final NASA FD001 Test Results
MAE:  18.95 cycles
RMSE: 25.27 cycles
R2:   0.6302
 Engine ID  Actual RUL  Predicted RUL  Absolute Error
         1         112         173.74           61.74
         2          98         146.44           48.44
         3          69          54.36           14.64
         4          82          76.47            5.53
         5          91          96.45            5.45
         6          93          92.31            0.69
         7          91         113.74           22.74
         8          95          73.56           21.44
         9         111         138.61           27.61
        10          96         111.67           15.67


### Step 7 — Inspect prediction errors

What: Count over- and underestimates and show the largest errors.

Why: Overestimating remaining life is especially important to monitor.

In [ ]:

# Analyze prediction errors — no retraining
error_analysis = test_predictions.copy()

error_analysis["Signed Error"] = (
    error_analysis["Predicted RUL"]
    - error_analysis["Actual RUL"]
)

print("Overestimated engines:",
      (error_analysis["Signed Error"] > 0).sum())

print("Underestimated engines:",
      (error_analysis["Signed Error"] < 0).sum())

print("\nMean signed error:",
      round(error_analysis["Signed Error"].mean(), 2))

print("\nLargest prediction errors:")
print(error_analysis.sort_values("Absolute Error", ascending=False).head(10).to_string(index=False))

Overestimated engines: 63
Underestimated engines: 37

Mean signed error: 9.67

Largest prediction errors:
 Engine ID  Actual RUL  Predicted RUL  Absolute Error  Signed Error
        15          83         166.29           83.29         83.29
        67          77         145.06           68.06         68.06
         1         112         173.74           61.74         61.74
        22         111         165.63           54.63         54.63
        78         107         157.28           50.28         50.28
         2          98         146.44           48.44         48.44
        41          18          66.25           48.25         48.25
        85         118         163.98           45.98         45.98
        45         114          73.15           40.85        -40.85
        14         107         146.95           39.95         39.95


### Step 8 — Save model and evaluation evidence

What: Export the trained model, feature order, predictions, and tuning tables.

Why: Allow teammates to reuse the model and review the results.

In [ ]:
joblib.dump({'model': final_rf, 'features': original_features}, OUTPUT_DIR / 'aeris_random_forest.joblib')
test_predictions.to_csv(OUTPUT_DIR / 'nasa_test_predictions.csv', index=False)
tuning_df.to_csv(OUTPUT_DIR / 'depth_leaf_cv.csv', index=False)
tree_results_df.to_csv(OUTPUT_DIR / 'tree_count_cv.csv', index=False)
pd.DataFrame([{'MAE': test_mae, 'RMSE': test_rmse, 'R2': test_r2}]).to_csv(OUTPUT_DIR / 'nasa_test_metrics.csv', index=False)
print('Saved model and results to:', OUTPUT_DIR)

Saved model and results to: /content/data/model_outputs


### Project status — Work in progress
This is our current Random Forest baseline, **not a fully optimized model**. We plan to improve RUL predictions and review errors across engines later.